# Universal Headless Kaggle Compute Server (Tailscale SSH)

This notebook turns any Kaggle GPU/TPU/CPU container into a **headless, persistent remote compute server**.
Your local agent (or you) can connect directly via SSH, sync code with `rsync`, run experiments, inspect logs, and fetch artifacts seamlessly.

---

### Quick Setup Instructions
1. **Add Auth Key**: In Kaggle top menu, click **Add-ons** $\to$ **Secrets** $\to$ add `TAILSCALE_AUTHKEY` with your generated auth key (`tskey-auth-...`).
2. **Enable Internet**: In the right sidebar, verify **Internet: On** is enabled.
3. **Select Accelerator**: In the right sidebar, select **GPU T4 x2**, **GPU P100**, or **None (CPU)** for free testing.
4. **Run**: Click the **Run Cell** (Play $\blacktriangleright$) button on the server cell below, or use **Save Version** $\to$ **Save & Run All (Commit)**.

---

### Connecting from your local machine
```bash
# 1. Interactive terminal session
ssh root@kaggle-server

# 2. Sync local project to Kaggle workspace
rsync -avz --exclude='.git' ./ root@kaggle-server:/kaggle/working/workspace/

# 3. Execute commands remotely
ssh root@kaggle-server "cd /kaggle/working/workspace && python train.py"

# 4. Gracefully terminate server and save outputs
ssh root@kaggle-server "stop-server"
```

In [ ]:
# --- 1. Environment & Hardware Diagnostics ---
import os
import sys
import shutil
import subprocess

print("=" * 60)
print("           SYSTEM & ACCELERATOR DIAGNOSTICS")
print("=" * 60)
print(f"Python Version: {sys.version.split()[0]}")

try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    if torch.cuda.is_available():
        count = torch.cuda.device_count()
        print(f"CUDA Available:  True ({count} device{'s' if count > 1 else ''})")
        for i in range(count):
            props = torch.cuda.get_device_properties(i)
            print(f"  Device {i}:     {props.name} ({props.total_memory / 1e9:.2f} GB VRAM)")
    else:
        print("CUDA Available:  False (Running on CPU)")
except ImportError:
    print("PyTorch not installed.")

total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Disk Space:      {free / (1024**3):.2f} GB free / {total / (1024**3):.2f} GB total")

try:
    subprocess.run(["curl", "-s", "-o", "/dev/null", "--max-time", "5", "https://www.google.com"], check=True)
    print("Internet:        Connected (Outbound traffic functional)")
except Exception:
    print("ERROR: Internet is DISABLED. Toggle 'Internet: On' in the Kaggle sidebar settings!")
print("=" * 60)

In [ ]:
# --- 2. Headless Server Daemon & Keepalive ---
import os
import sys
import time
import datetime
import subprocess
import zipfile
from kaggle_secrets import UserSecretsClient

# 1. Fetch Tailscale AuthKey securely
try:
    secrets = UserSecretsClient()
    TS_AUTHKEY = secrets.get_secret("TAILSCALE_AUTHKEY")
    if not TS_AUTHKEY or not TS_AUTHKEY.startswith("tskey-auth-"):
        raise ValueError("Secret 'TAILSCALE_AUTHKEY' does not appear to be a valid Tailscale key (should start with 'tskey-auth-').")
except Exception as e:
    print(f"\n[!] ERROR loading TAILSCALE_AUTHKEY: {e}")
    print("    Make sure you added 'TAILSCALE_AUTHKEY' in Add-ons -> Secrets and enabled the secret checkbox!")
    raise

HOSTNAME = "kaggle-server"
WORKSPACE_DIR = "/kaggle/working/workspace"
os.makedirs(WORKSPACE_DIR, exist_ok=True)
os.makedirs("/var/run/tailscale", exist_ok=True)

# 2. Fast install via official static binaries (avoids heavy apt-get churn)
print("\n[1/3] Setting up Tailscale & utilities...")
install_script = """
if ! command -v tailscaled &> /dev/null; then
    curl -fsSL https://pkgs.tailscale.com/stable/tailscale_latest_amd64.tgz | tar -xz -C /tmp/
    cp /tmp/tailscale_*_amd64/tailscale /usr/local/bin/
    cp /tmp/tailscale_*_amd64/tailscaled /usr/local/bin/
    chmod +x /usr/local/bin/tailscale /usr/local/bin/tailscaled
fi

# Install helper script for graceful stop
cat << 'EOF' > /usr/local/bin/stop-server
#!/bin/bash
echo '[*] Signaling server loop to stop cleanly...'
touch /kaggle/working/.stop_server
EOF
chmod +x /usr/local/bin/stop-server

# Install helper script for GPU query
cat << 'EOF' > /usr/local/bin/gpu-info
#!/bin/bash
if command -v nvidia-smi &> /dev/null; then
    nvidia-smi --query-gpu=index,name,utilization.gpu,utilization.memory,memory.used,memory.total --format=csv,nounits
else
    echo 'Running on CPU (no GPU detected).'
fi
EOF
chmod +x /usr/local/bin/gpu-info
"""
subprocess.run(install_script, shell=True, check=True)

# 3. Start tailscaled in userspace mode with in-memory state
print("[2/3] Starting tailscaled daemon (userspace mode, in-memory state)...")
subprocess.run(["pkill", "-9", "tailscaled"], stderr=subprocess.DEVNULL)
time.sleep(1)

log_file = open("/tmp/tailscaled.log", "w")
subprocess.Popen(
    [
        "tailscaled",
        "--tun=userspace-networking",
        "--state=mem:",
        "--socket=/var/run/tailscale/tailscaled.sock",
    ],
    stdout=log_file,
    stderr=log_file,
    start_new_session=True,
)
time.sleep(2)

# 4. Authenticate node to Tailscale
print("[3/3] Authenticating to Tailscale...")
subprocess.run(
    [
        "tailscale",
        "up",
        f"--authkey={TS_AUTHKEY}",
        f"--hostname={HOSTNAME}",
        "--ssh",
        "--accept-routes=false",
        "--netfilter-mode=off",
    ],
    check=True,
)

# Fetch assigned IP
ip_res = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True)
ts_ip = ip_res.stdout.strip() if ip_res.returncode == 0 else "Unknown"

print("\n" + "=" * 60)
print("           KAGGLE REMOTE SERVER IS ONLINE!")
print("=" * 60)
print(f"  Hostname:            {HOSTNAME}")
print(f"  Tailscale IPv4:      {ts_ip}")
print(f"  Workspace:           {WORKSPACE_DIR}")
print("-" * 60)
print("  Commands to use locally:")
print(f"    ssh root@{HOSTNAME}")
print(f"    rsync -avz --exclude='.git' ./ root@{HOSTNAME}:{WORKSPACE_DIR}/")
print(f"    ssh root@{HOSTNAME} \"stop-server\"")
print("=" * 60 + "\n")

# 5. Heartbeat & keepalive loop in the same execution context
STOP_TRIGGER = "/kaggle/working/.stop_server"
if os.path.exists(STOP_TRIGGER):
    os.remove(STOP_TRIGGER)

start_time = datetime.datetime.now()
print("[+] Heartbeat loop active. Ready for SSH connections.")

try:
    while True:
        if os.path.exists(STOP_TRIGGER):
            print("\n[!] Stop trigger detected. Shutting down gracefully...")
            break
        
        elapsed_sec = int((datetime.datetime.now() - start_time).total_seconds())
        mins = elapsed_sec // 60
        secs = elapsed_sec % 60
        print(f"[*] Server online | Uptime: {mins:02d}m {secs:02d}s | Active listener OK", flush=True)
        
        time.sleep(30)

except KeyboardInterrupt:
    print("\n[!] KeyboardInterrupt received.")

# Auto-package results on exit
results_dir = os.path.join(WORKSPACE_DIR, "results")
bundle_path = "/kaggle/working/server_artifacts.zip"
if os.path.exists(results_dir):
    print(f"Packaging results from {results_dir} into {bundle_path}...")
    with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(results_dir):
            for f in files:
                fp = os.path.join(root, f)
                arcname = os.path.relpath(fp, "/kaggle/working")
                zf.write(fp, arcname)
    print(f"[+] Created artifact bundle: {bundle_path}")

print("[+] Clean shutdown completed. Exiting notebook.")